In [1]:
import configparser
import os
import json
import time
import re
from openai import OpenAI, RateLimitError, APITimeoutError, APIConnectionError, InternalServerError

# Inicialització del client
cfg_path = "/media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/utils/config.ini"
cfg = configparser.ConfigParser()
cfg.read(cfg_path)
if 'OPENAI' in cfg and 'KEY' in cfg['OPENAI']:
    key = cfg['OPENAI']['KEY']
    font = f"config.ini:{cfg_path}"
base_url = os.environ.get("OPENAI_BASE_URL")  # opcional: servidors locals
if base_url:
    font += f" (base_url={base_url})"
client = OpenAI(api_key=key, base_url=base_url)

MODEL = 'gpt-4o-mini' 

ruta_diccionari = "/media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/diccionari_fonetic_castellano_small.json"
try:
    with open(ruta_diccionari, 'r', encoding='utf-8') as f:
        diccionari_fonetic = json.load(f)
except FileNotFoundError:
    raise RuntimeError(f"Error: No se ha encontrado el archivo en la ruta '{ruta_diccionari}'. Revisa que el path sea correcto.")

OUTPUT_FILE = "dataset_entitats_whisper.jsonl"

# Errores recuperables (rate limit / red / servidor caído): se reintentan con backoff.
# Cualquier otro error (contenido bloqueado, JSON inválido, etc.) se propaga de inmediato,
# porque reintentar no lo va a arreglar.
ERRORES_RECUPERABLES = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)


def call_with_retry(fn, *args, retries=3, backoff_base=2, **kwargs):
    """Ejecuta fn(*args, **kwargs) reintentando con backoff exponencial solo ante errores recuperables."""
    for intent in range(retries):
        try:
            return fn(*args, **kwargs)
        except ERRORES_RECUPERABLES as e:
            if intent == retries - 1:
                raise
            espera = backoff_base ** intent
            print(f"  Error recuperable ({type(e).__name__}), reintento {intent + 1}/{retries} en {espera}s...")
            time.sleep(espera)


def chunked(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [2]:
SCHEMA_CONTEXT_BATCH = {
    "type": "object",
    "properties": {
        "entidades": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entidad": {"type": "string", "description": "La entidad tal como se ha pedido, sin modificar."},
                    "tipo_entidad": {"type": "string", "description": "Persona, Organización, Lugar, Tecnología, etc."},
                    "contexto_periodistico": {"type": "string", "description": "Breve descripción de 1 línea para dar contexto a un periodista."}
                },
                "required": ["entidad", "tipo_entidad", "contexto_periodistico"],
                "additionalProperties": False
            }
        }
    },
    "required": ["entidades"],
    "additionalProperties": False
}


def deduir_contexts_batch(entitats):
    """Deduce el contexto de una lista de entidades en una única llamada (en vez de una llamada por entidad)."""
    prompt_sys = "Eres un documentalista de informativos de televisión. Define cada entidad de la lista."
    prompt_user = (
        "Entidades: " + ", ".join(f"'{e}'" for e in entitats) +
        ". Devuelve tipo y contexto breve para cada una, respetando el campo 'entidad' exactamente como se ha escrito."
    )

    resposta = call_with_retry(
        client.chat.completions.create,
        model=MODEL,
        temperature=0.2,  # Molt baix perquè volem dades objectives
        messages=[
            {"role": "system", "content": prompt_sys},
            {"role": "user", "content": prompt_user}
        ],
        response_format={"type": "json_schema", "json_schema": {"name": "context_batch_schema", "strict": True, "schema": SCHEMA_CONTEXT_BATCH}}
    )

    items = json.loads(resposta.choices[0].message.content)["entidades"]
    return {item["entidad"]: item for item in items}

In [ ]:
SCHEMA_FRASES = {
    "type": "object",
    "properties": {
        "frases": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "texto": {
                        "type": "string",
                        "description": "Frase con ortografía real. La entidad puede aparecer flexionada (género, número o conjugación) para que la frase sea gramaticalmente correcta."
                    },
                    "texto_tts": {
                        "type": "string",
                        "description": "La misma frase, sustituyendo únicamente la entidad por su transcripción fonética, flexionada igual que en 'texto'."
                    }
                },
                "required": ["texto", "texto_tts"],
                "additionalProperties": False
            },
            "description": "Lista de frases generadas, cada una con su versión ortográfica y su versión fonética para TTS."
        }
    },
    "required": ["frases"],
    "additionalProperties": False
}


def generar_batch_frases(entitat, fonetica, context, quantitat, estil):
    prompt_sys = f"""Eres un guionista de informativos de RTVE (Telediario).
Debes generar {quantitat} frases independientes que usen la entidad '{entitat}', respetando la concordancia gramatical de la frase (género, número y tiempo verbal si es un verbo o participio). La entidad puede aparecer flexionada (p.ej. en plural, en femenino, conjugada) siempre que se reconozca la misma raíz; NUNCA sustituida por otra palabra distinta.

Contexto de la entidad: {context['tipo_entidad']} - {context['contexto_periodistico']}
Estilo periodístico requerido para este lote: {estil}

Para cada frase entrega dos campos:
- "texto": ortografía real, con la entidad en la forma gramatical que corresponda.
- "texto_tts": la MISMA frase, sustituyendo la entidad por su transcripción fonética, adaptada a la misma forma gramatical usada en "texto". Como referencia, la transcripción fonética de la forma base '{entitat}' es '{fonetica}'; aplica el mismo criterio de transliteración al adaptarla."""

    resposta = call_with_retry(
        client.chat.completions.create,
        model=MODEL,
        temperature=0.7,  # Més alt per forçar creativitat sintàctica
        messages=[
            {"role": "system", "content": prompt_sys},
            {"role": "user", "content": "Genera las frases ahora."}
        ],
        response_format={"type": "json_schema", "json_schema": {"name": "frases_schema", "strict": True, "schema": SCHEMA_FRASES}}
    )

    return json.loads(resposta.choices[0].message.content)["frases"]

In [4]:
# Si alguna entidad con significado ambigüo, el archivo de contextos habria que revisarlo a mano
ARCHIVO_AUDITORIA = "contextos_auditables.json"
BATCH_SIZE_CONTEXTO = 10  # nº de entidades por llamada a la API en Fase 1

diccionari_contextos = {}
entitats = list(diccionari_fonetic.keys())

print("Fase 1: Deducción de contextos...")

for lot in chunked(entitats, BATCH_SIZE_CONTEXTO):
    try:
        contexts = deduir_contexts_batch(lot)
    except Exception as e:
        print(f"Error al deducir contexto para el lote {lot}: {e}")
        contexts = {}

    for entitat in lot:
        fonetica = diccionari_fonetic[entitat]
        info = contexts.get(entitat)

        if info is not None:
            diccionari_contextos[entitat] = {
                "fonetica": fonetica,
                "tipo_entidad": info["tipo_entidad"],
                "contexto_periodistico": info["contexto_periodistico"]
            }
            print(f"Contexto generado para: {entitat} -> {info['tipo_entidad']}")
        else:
            print(f"Sin contexto para '{entitat}': revisar manualmente en '{ARCHIVO_AUDITORIA}'.")
            # Si falla, dejamos un placeholder para que lo rellenes tú a mano luego
            diccionari_contextos[entitat] = {
                "fonetica": fonetica,
                "tipo_entidad": "ERROR_API_REVISAR",
                "contexto_periodistico": "ERROR_API_REVISAR"
            }

# Guardamos el archivo con indentación (indent=4) para que sea legible
with open(ARCHIVO_AUDITORIA, 'w', encoding='utf-8') as f_out:
    json.dump(diccionari_contextos, f_out, indent=4, ensure_ascii=False)

print(f"\nFASE 1 COMPLETADA. El archivo '{ARCHIVO_AUDITORIA}' se ha guardado.")
print("REVISIÓN MANUAL: CORREGIR los contextos equivocados antes de continuar.")

Fase 1: Deducción de contextos...
Contexto generado para: Hollywood -> Lugar
Contexto generado para: Startup -> Organización
Contexto generado para: Mbappé -> Persona
Contexto generado para: Hackeado -> Tecnología
Contexto generado para: Spoilear -> Acción

FASE 1 COMPLETADA. El archivo 'contextos_auditables.json' se ha guardado.
REVISIÓN MANUAL: CORREGIR los contextos equivocados antes de continuar.


In [5]:
import concurrent.futures

ARCHIVO_AUDITORIA = "contextos_auditables.json"
OUTPUT_FILE = "dataset_entitats_whisper.jsonl"
ARCHIVO_DESCARTES = "frases_descartadas.jsonl"

estils_batches = [
    "Titular de última hora (frases cortas y directas)",
    "Crónica detallada del corresponsal (frases más largas y descriptivas)",
    "Entradilla del presentador en plató (tono formal e introductorio)",
    "Declaraciones en una rueda de prensa o debate (estilo más hablado)",
    "Noticia breve de sección de impacto (tono urgente)"
]

FRASES_PER_BATCH = 10
MAX_WORKERS = 4  # llamadas a la API en paralelo durante la Fase 2


def normalizar(text):
    return re.sub(r"\s+", " ", text).strip().lower()


def raiz_entitat(entitat):
    """Raíz mínima de la entidad para validar que una forma flexionada la reconoce.
    Es una heurística (no un análisis morfológico real), pero basta para detectar
    cuando el LLM se ha ido por completo de la entidad pedida."""
    e = entitat.lower()
    for sufijo in ("ar", "er", "ir", "os", "as", "es", "o", "a", "e", "s"):
        if len(e) - len(sufijo) >= 4 and e.endswith(sufijo):
            return e[: -len(sufijo)]
    return e


# 1. Cargar los contextos ya auditados
try:
    with open(ARCHIVO_AUDITORIA, 'r', encoding='utf-8') as f:
        datos_auditados = json.load(f)
except FileNotFoundError:
    raise RuntimeError(f"Error: No encuentro '{ARCHIVO_AUDITORIA}'. Ejecuta la Fase 1 primero.")

# 2. Cargar estado existente para deduplicar y para poder relanzar el notebook
#    sin duplicar todo lo ya generado (idempotencia por combinación entidad+estilo).
frases_vistas = set()
generadas_por_combo = {}

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            registre = json.loads(linea)
            frases_vistas.add(normalizar(registre["raw_text"]))
            combo = (registre["entity"], registre["style"])
            generadas_por_combo[combo] = generadas_por_combo.get(combo, 0) + 1

print("🚀 Iniciando Fase 2: Generación masiva de dataset...")

# 3. Preparar las tareas pendientes, saltando combinaciones ya completas
tareas = []
for entitat, info in datos_auditados.items():
    if info['tipo_entidad'] == "ERROR_API_REVISAR":
        print(f"Saltando '{entitat}': Tiene un error sin revisar en el archivo de auditoría.")
        continue

    context = {"tipo_entidad": info["tipo_entidad"], "contexto_periodistico": info["contexto_periodistico"]}
    fonetica = info["fonetica"]

    for estil in estils_batches:
        ja_generadas = generadas_por_combo.get((entitat, estil), 0)
        if ja_generadas >= FRASES_PER_BATCH:
            print(f"Saltando '{entitat}' / '{estil[:20]}...': ya hay {ja_generadas} frases (idempotencia).")
            continue
        tareas.append((entitat, fonetica, context, estil))

# 4. Lanzar las llamadas a la API en paralelo. Solo se paraleliza la parte de red;
#    la escritura del fichero y el recuento se hacen en el hilo principal según van
#    llegando los resultados, así no hace falta ningún lock.
contadores = {}

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futuros = {
        executor.submit(generar_batch_frases, entitat, fonetica, context, FRASES_PER_BATCH, estil): (entitat, estil)
        for entitat, fonetica, context, estil in tareas
    }

    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out, open(ARCHIVO_DESCARTES, 'a', encoding='utf-8') as f_desc:
        for futuro in concurrent.futures.as_completed(futuros):
            entitat, estil = futuros[futuro]
            stats = contadores.setdefault(entitat, {"ok": 0, "sin_entidad": 0, "duplicada": 0, "error_batch": 0})

            try:
                frases_generadas = futuro.result()
            except Exception as e:
                print(f" Error en batch '{estil}' de '{entitat}': {e}")
                stats["error_batch"] += 1
                continue

            raiz = raiz_entitat(entitat)

            for frase in frases_generadas:
                texto = frase.get("texto", "")
                texto_tts = frase.get("texto_tts", "")

                if raiz not in texto.lower():
                    stats["sin_entidad"] += 1
                    f_desc.write(json.dumps(
                        {"entity": entitat, "style": estil, "texto": texto, "motivo": "no_contiene_entidad"},
                        ensure_ascii=False) + "\n")
                    continue

                clave = normalizar(texto)
                if clave in frases_vistas:
                    stats["duplicada"] += 1
                    f_desc.write(json.dumps(
                        {"entity": entitat, "style": estil, "texto": texto, "motivo": "duplicada"},
                        ensure_ascii=False) + "\n")
                    continue

                frases_vistas.add(clave)
                registre = {
                    "entity": entitat,
                    "raw_text": texto,
                    "tts_text": texto_tts,
                    "style": estil
                }
                f_out.write(json.dumps(registre, ensure_ascii=False) + '\n')
                stats["ok"] += 1

            print(f"[{entitat}] Batch '{estil[:15]}...' -> {stats['ok']} ok / {stats['sin_entidad']} sin entidad / {stats['duplicada']} duplicadas (acumulado)")

print("\nResumen Fase 2:")
for entitat, stats in contadores.items():
    print(f"  {entitat}: {stats['ok']} guardadas, {stats['sin_entidad']} descartadas sin entidad, "
          f"{stats['duplicada']} duplicadas, {stats['error_batch']} batches fallidos")

print(f"\nProceso completado. Datos de entrenamiento guardados en {OUTPUT_FILE}")
print(f"Frases descartadas (auditoría) guardadas en {ARCHIVO_DESCARTES}")

🚀 Iniciando Fase 2: Generación masiva de dataset...
[Hollywood] Batch 'Titular de últi...' -> 10 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Hollywood] Batch 'Entradilla del ...' -> 20 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Hollywood] Batch 'Declaraciones e...' -> 30 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Hollywood] Batch 'Crónica detalla...' -> 40 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Startup] Batch 'Titular de últi...' -> 10 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Hollywood] Batch 'Noticia breve d...' -> 50 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Startup] Batch 'Crónica detalla...' -> 20 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Startup] Batch 'Entradilla del ...' -> 30 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Startup] Batch 'Declaraciones e...' -> 40 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Startup] Batch 'Noticia breve d...' -> 50 ok / 0 sin entidad / 0 duplicadas (acumulado)
[Mbappé] Batch 'Titular de últi...' -> 10 ok / 0

## Fase 3: entitats secundàries

**El problema.** La Fase 2 només transcriu fonèticament l'**entitat objectiu** de cada frase.
La resta d'entitats que el model introdueix mentre redacta queden sense tocar:

| | |
|---|---|
| `raw_text` | El **FC Barcelona** confía en **Lamine Yamal** como una de sus futuras estrellas. |
| `tts_text` (Fase 2) | El **FC Barcelona** confía en **Lamín Yamal** como una de sus futuras estrellas. |

`Lamine Yamal` sí que s'ha tractat (és l'entitat objectiu), però `FC Barcelona` arriba a
OmniVoice tal qual i es pot llegir malament. Passa el mateix amb sigles, anglicismes i
símbols que apareixen "de rebot" a les frases.

**La solució: un post-procés en tres passos**, no un canvi al prompt de generació. Fer-ho
així també arregla les frases **ja generades**, sense haver de tornar-les a demanar a l'API:

1. **3a Detecció** — una passada de LLM sobre `raw_text` que llista els fragments
   impronunciables *que no són* l'entitat objectiu.
2. **3b Transcripció** — els fragments únics passen **un sol cop** per les regles de
   respelling de `dictionary.ipynb` → diccionari secundari auditable.
3. **3c Aplicació** — find-and-replace determinista sobre `tts_text`.

La clau és separar detecció de transcripció: com que cada entitat es transcriu una única
vegada i després només se substitueix, `FC Barcelona` sonarà **igual a totes les frases**.
Si demanéssim la transcripció dins de cada crida de generació, cada frase podria rebre'n
una variant lleugerament diferent.

### Fase 3a: detecció de candidates

Sortida: `entitats_secundaries_candidates.json` (entitat, freqüència, frase d'exemple).
Es dedupliquen les frases abans de cridar l'API i es descarta el soroll típic del model
(fragments que no són literalment al text, l'entitat objectiu, números solts).


In [ ]:
import concurrent.futures
from pathlib import Path
from collections import Counter

ROOT = Path("/media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset")

# --- Entrada: el dataset ya generado por la Fase 2 ---
DATASET_IN = ROOT / "lab/outputs/frases/dataset_entitats_whisper_1.jsonl"

# --- Salidas de la Fase 3 ---
CANDIDATS_OUT = ROOT / "lab/outputs/frases/entitats_secundaries_candidates.json"
DICCIONARI_SECUNDARI = ROOT / "lab/entitats/diccionaris/diccionari_fonetic_secundari.json"
DATASET_OUT = ROOT / "lab/outputs/frases/dataset_entitats_whisper_final.jsonl"

# Diccionario principal (Pas B): sus transcripciones mandan sobre las secundarias.
DICCIONARI_PRINCIPAL = ROOT / "lab/entitats/diccionaris/diccionari_fonetic_castellano.json"

IDIOMA_TTS = 'Castellano'   # Debe coincidir con el idioma usado en `dictionary.ipynb`
MODEL_DETECCIO = MODEL      # gpt-4o-mini basta para detectar fragmentos
MODEL_FONETICA = 'gpt-4o'   # Para transcribir usamos el mismo modelo que `dictionary.ipynb`

BATCH_DETECCIO = 20         # frases por llamada
MAX_WORKERS_DETECCIO = 4
MIDA_LOT_FONETICA = 40      # entidades por llamada


def clau(text):
    """Clave de comparación: minúsculas y espacios normalizados. Sirve para deduplicar
    variantes de la misma entidad ('FC Barcelona' / 'fc  barcelona') sin perder la forma
    original, que es la que necesitamos para buscarla en el texto."""
    return re.sub(r"\s+", " ", text).strip().lower()


def patro_entitat(entitat):
    """Patrón regex que localiza una entidad dentro de un texto. Es la única definición
    de 'dónde empieza y acaba una entidad' del notebook: la usan tanto la detección
    (Fase 3a) como la sustitución (Fase 3c).

    - Los espacios se convierten en \\s+ para tolerar espaciado irregular.
    - Los límites de palabra solo se ponen si el borde es alfanumérico: entidades como
      '%' o 'km/h' no tienen frontera \\w a ese lado y el límite las haría no encontrables.
    """
    cos = r"\s+".join(re.escape(part) for part in entitat.split())
    pre = r"(?<!\w)" if entitat[:1].isalnum() else ""
    post = r"(?!\w)" if entitat[-1:].isalnum() else ""
    return f"{pre}{cos}{post}"


def conte_entitat(text, entitat):
    """¿Aparece `entitat` dentro de `text` como fragmento completo? Con un simple
    `in` tendríamos falsos positivos con entidades cortas: 'UE' está dentro de
    'Nueva York' como subcadena, pero no como entidad."""
    if not entitat:
        return False
    return re.search(patro_entitat(entitat), text) is not None


SCHEMA_DETECCIO = {
    "type": "object",
    "properties": {
        "frases": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "El mismo id que se ha recibido para esa frase."},
                    "fragmentos": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Fragmentos copiados LITERALMENTE de la frase que un TTS castellano podría leer mal. Lista vacía si no hay ninguno."
                    }
                },
                "required": ["id", "fragmentos"],
                "additionalProperties": False
            }
        }
    },
    "required": ["frases"],
    "additionalProperties": False
}

PROMPT_DETECCIO = f"""Eres un lingüista que prepara textos de informativos para un sistema Text-to-Speech en {IDIOMA_TTS}.

Recibirás una lista de frases. En cada una hay una 'entidad_objetivo' que YA está tratada: IGNÓRALA por completo.
Tu tarea es localizar el RESTO de fragmentos que un TTS en {IDIOMA_TTS} podría pronunciar mal.

Marca:
- Nombres propios extranjeros o de lectura difícil (personas, clubes, marcas, topónimos).
- Siglas y acrónimos (FC Barcelona, UE, PSOE, NASA, ONU).
- Extranjerismos y anglicismos (startup, hacker, prime time, streaming).
- Símbolos y unidades (%, ºC, km/h, €).

NO marques:
- La entidad_objetivo ni ninguna variante suya.
- Palabras castellanas corrientes, aunque sean poco frecuentes.
- Nombres propios castellanos que se leen de forma evidente (Madrid, Pedro, Valencia).
- Números sueltos escritos en cifras, salvo que formen parte de una unidad o símbolo.

Reglas de salida:
- Copia cada fragmento EXACTAMENTE como aparece en la frase (mismas mayúsculas, tildes y espacios). No lo normalices ni lo traduzcas.
- Si el fragmento aparece flexionado (plural, femenino, conjugado), cópialo tal cual está flexionado en la frase.
- Si una sigla acompaña a un nombre, devuelve el fragmento completo ("FC Barcelona", no solo "FC").
- Devuelve una entrada por cada frase recibida, con su mismo id. Si una frase no tiene nada que marcar, devuelve una lista vacía.
"""


def detectar_lot(lot):
    """Detecta fragmentos problemáticos en un lote de frases.
    `lot` es una lista de (id, entidad_objetivo, raw_text).
    Devuelve [(id, [fragmentos...]), ...]."""
    payload = [{"id": idx, "entidad_objetivo": ent, "frase": text} for idx, ent, text in lot]

    resposta = call_with_retry(
        client.chat.completions.create,
        model=MODEL_DETECCIO,
        temperature=0,  # Determinismo: detectar no es una tarea creativa
        seed=42,
        messages=[
            {"role": "system", "content": PROMPT_DETECCIO},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
        ],
        response_format={"type": "json_schema", "json_schema": {"name": "deteccio_secundaries", "strict": True, "schema": SCHEMA_DETECCIO}},
    )

    resultat = json.loads(resposta.choices[0].message.content)["frases"]
    return [(item["id"], item["fragmentos"]) for item in resultat]


# 1. Cargar el dataset y deduplicar frases: no hace falta preguntar dos veces por la
#    misma frase (y en un dataset con 50 frases por entidad se repiten formulaciones).
registres = []
with open(DATASET_IN, 'r', encoding='utf-8') as f:
    for linia in f:
        linia = linia.strip()
        if linia:
            registres.append(json.loads(linia))

frases_uniques = {}  # clave normalizada -> (id, entidad, raw_text)
for registre in registres:
    k = clau(registre["raw_text"])
    if k not in frases_uniques:
        frases_uniques[k] = (len(frases_uniques), registre["entity"], registre["raw_text"])

tasques = list(frases_uniques.values())
per_id = {idx: (ent, text) for idx, ent, text in tasques}

print(f"Frases cargadas: {len(registres)} ({len(tasques)} únicas)")
print(f"Fase 3a: detectando entidades secundarias con {MODEL_DETECCIO}...")

# 2. Detección en paralelo (solo la parte de red; la agregación va en el hilo principal,
#    igual que en la Fase 2, así no hace falta ningún lock)
comptador = Counter()          # clave normalizada -> nº de frases donde aparece
forma_canonica = {}            # clave normalizada -> forma tal como aparece en el texto
exemple = {}                   # clave normalizada -> frase de ejemplo
descartats = Counter()

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS_DETECCIO) as executor:
    futurs = [executor.submit(detectar_lot, lot) for lot in chunked(tasques, BATCH_DETECCIO)]

    for n, futur in enumerate(concurrent.futures.as_completed(futurs), start=1):
        try:
            resultats = futur.result()
        except Exception as e:
            print(f"  Error en un lote de detección: {e}")
            continue

        for idx, fragments in resultats:
            if idx not in per_id:
                continue  # el modelo se ha inventado un id
            entitat, text = per_id[idx]

            for fragment in fragments:
                fragment = fragment.strip()
                k = clau(fragment)

                # Guardas contra el ruido típico del modelo:
                if not k or k.isdigit():
                    descartats["buit_o_numero"] += 1
                elif fragment not in text:
                    # No está literalmente en la frase: alucinación o normalización del
                    # modelo. Sin coincidencia exacta no podríamos sustituirlo después.
                    descartats["no_literal"] += 1
                elif conte_entitat(k, clau(entitat)) or conte_entitat(clau(entitat), k):
                    # Es la entidad objetivo (o la contiene): ya la trata la Fase 2.
                    descartats["entitat_objectiu"] += 1
                else:
                    comptador[k] += 1
                    forma_canonica.setdefault(k, fragment)
                    exemple.setdefault(k, text)

        print(f"  Lote {n}/{len(futurs)} procesado ({len(comptador)} candidatas únicas)")

# 3. Guardar candidatas ordenadas por frecuencia: las más frecuentes son las que más
#    impacto tienen en el dataset final y las primeras que conviene revisar.
candidates = [
    {"entitat": forma_canonica[k], "frequencia": n, "exemple": exemple[k]}
    for k, n in comptador.most_common()
]

CANDIDATS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(CANDIDATS_OUT, 'w', encoding='utf-8') as f:
    json.dump(candidates, f, indent=2, ensure_ascii=False)

print(f"\nCandidatas detectadas: {len(candidates)} -> {CANDIDATS_OUT}")
print(f"Descartadas: {dict(descartats)}")
print("\nTop 20 por frecuencia:")
for c in candidates[:20]:
    print(f"  {c['frequencia']:5d}x  {c['entitat']}")


### Fase 3b: transcripció fonètica de les entitats secundàries

Les candidates úniques passen **una sola vegada** per les mateixes regles de respelling que
`dictionary.ipynb` (Pas B). Aquí és on guanyem la consistència: com que cada entitat es
transcriu un únic cop i després s'aplica per find-and-replace, *"FC Barcelona"* sonarà
igual a totes les frases del dataset.

Les entitats que ja són al diccionari principal **no** es tornen a demanar: aquell ja està
auditat i té prioritat.

Sortida: `lab/entitats/diccionaris/diccionari_fonetic_secundari.json` → **revisió manual** abans de
la Fase 3c, igual que es fa amb el diccionari principal.


In [ ]:
# Mismas reglas de respelling que `dictionary.ipynb` (Pas B). Se mantienen aquí en vez de
# importarlas porque aquel notebook no expone un módulo; si algún día se extraen a
# `utils/`, este prompt debería ser el mismo objeto.
SYSTEM_PROMPT_FONETICA = f"""Eres un lingüista experto en fonética y sistemas Text-to-Speech (TTS).
Vas a recibir una lista de entidades (nombres propios, acrónimos, extranjerismos, símbolos)
extraídas de frases de informativos. Devuelve su adaptación fonética para que un modelo TTS
configurado en {IDIOMA_TTS} las lea correctamente y con naturalidad, como un presentador de noticias.

Reglas de integridad (obligatorias):
- Devuelve exactamente una entrada por cada entidad de entrada, ni más ni menos.
- Conserva el orden de la lista de entrada.
- Nunca traduzcas, abrevies, inventes, omitas ni dividas una entidad en varias.
- Si una entidad ya se pronunciaría correctamente tal cual, devuélvela SIN MODIFICAR.
  Es importante: solo queremos tocar lo que de verdad se leería mal.

Reglas de reescritura fonética:
1. Siglas deletreadas (ej. UE, FMI): separa cada letra con un espacio y mantenlas en mayúscula ("U E", "F M I").
2. Acrónimos léxicos (ej. PSOE, OTAN): capitaliza solo la primera letra y ajusta la acentuación si la pronunciación real no coincide con la acentuación por defecto ("Psoe", "Otán").
3. Extranjerismos y nombres propios complejos: aplica respelling fonético con la ortografía literal de {IDIOMA_TTS}. Si empieza por un grupo consonántico imposible en {IDIOMA_TTS} (ej. "Mb", "Ng", "Pf"), añade una vocal de apoyo o simplifica el grupo.
4. Símbolos y unidades (ej. ºC, %, km/h): expándelos a su forma leída completa en palabras ("grados Celsius", "por ciento", "kilómetros por hora").
5. Entidades mixtas sigla + nombre (ej. "FC Barcelona"): aplica a cada parte la regla que le toque, manteniendo la entidad como una sola cadena ("Efe Ce Barcelona").

Ejemplos orientativos:
- "Wall Street" -> "Guol estrit"
- "Lamine Yamal" -> "Lamín Yamal"
- "Mbappé" -> "Embapé"
- "Junior" -> "Yúnior"
"""

SCHEMA_FONETICA = {
    "type": "object",
    "properties": {
        "diccionari": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entitat_original": {"type": "string", "description": "La entidad tal como se ha recibido, sin modificar."},
                    "transcripcio_fonetica": {"type": "string", "description": "La reescritura fonética según las reglas, o la entidad sin cambios si ya se lee bien."}
                },
                "required": ["entitat_original", "transcripcio_fonetica"],
                "additionalProperties": False
            }
        }
    },
    "required": ["diccionari"],
    "additionalProperties": False
}


def transcriure_lot(entitats):
    """Transcribe fonéticamente un lote de entidades. Devuelve {entidad: fonética}."""
    resposta = call_with_retry(
        client.chat.completions.create,
        model=MODEL_FONETICA,
        temperature=0,  # Determinismo: queremos la misma salida si se relanza
        seed=42,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_FONETICA},
            {"role": "user", "content": f"Entidades a procesar:\n{json.dumps(entitats, ensure_ascii=False)}"},
        ],
        response_format={"type": "json_schema", "json_schema": {"name": "foneticas_secundarias", "strict": True, "schema": SCHEMA_FONETICA}},
    )
    items = json.loads(resposta.choices[0].message.content)["diccionari"]
    return {item["entitat_original"]: item["transcripcio_fonetica"] for item in items}


# 1. Cargar candidatas y descartar las que ya están en el diccionario principal
#    (esas ya se auditaron en el Pas B y su transcripción manda).
with open(CANDIDATS_OUT, 'r', encoding='utf-8') as f:
    candidates = json.load(f)

with open(DICCIONARI_PRINCIPAL, 'r', encoding='utf-8') as f:
    diccionari_principal = json.load(f)

claus_principals = {clau(k) for k in diccionari_principal}
pendents = [c["entitat"] for c in candidates if clau(c["entitat"]) not in claus_principals]

# 2. Reanudable: si ya hay diccionario secundario, solo pedimos lo que falta.
diccionari_secundari = {}
if DICCIONARI_SECUNDARI.exists():
    with open(DICCIONARI_SECUNDARI, 'r', encoding='utf-8') as f:
        diccionari_secundari = json.load(f)
    pendents = [e for e in pendents if e not in diccionari_secundari]
    print(f"Diccionario secundario existente: {len(diccionari_secundari)} entradas ya transcritas.")

print(f"Entidades a transcribir: {len(pendents)}")

for lot in chunked(pendents, MIDA_LOT_FONETICA):
    try:
        resultat = transcriure_lot(lot)
    except Exception as e:
        print(f"Error transcribiendo el lote {lot[:3]}...: {e}")
        continue

    faltants = [e for e in lot if e not in resultat]
    if faltants:
        print(f"  Aviso: el modelo no ha devuelto {len(faltants)} entidades del lote: {faltants}")
    diccionari_secundari.update({e: resultat[e] for e in lot if e in resultat})
    print(f"  Lote de {len(lot)} procesado (acumulado: {len(diccionari_secundari)})")

DICCIONARI_SECUNDARI.parent.mkdir(parents=True, exist_ok=True)
with open(DICCIONARI_SECUNDARI, 'w', encoding='utf-8') as f:
    json.dump(diccionari_secundari, f, indent=2, ensure_ascii=False)

# 3. Resumen: separamos lo que el modelo ha decidido dejar igual (no se sustituirá)
#    de lo que sí cambia, que es lo que hay que revisar a mano.
canvien = {k: v for k, v in diccionari_secundari.items() if clau(k) != clau(v)}
sense_canvi = [k for k in diccionari_secundari if clau(k) == clau(diccionari_secundari[k])]

print(f"\nDiccionario secundario guardado en: {DICCIONARI_SECUNDARI}")
print(f"  {len(canvien)} entidades con transcripción distinta (se aplicarán)")
print(f"  {len(sense_canvi)} entidades que el modelo deja igual (no se tocarán)")
print("\nTranscripciones propuestas:")
for k, v in sorted(canvien.items()):
    print(f"  {k!r} -> {v!r}")

print("\n>>> REVISIÓN MANUAL: corrige o borra entradas en el JSON antes de la Fase 3c.")


### Fase 3c: aplicació determinista al dataset

Un cop revisat `diccionari_fonetic_secundari.json`, apliquem les substitucions sobre
`tts_text` amb un simple find-and-replace. **Aquí no hi ha LLM**: la mateixa entitat rep
sempre la mateixa transcripció, a totes les frases del dataset.

Detalls de la substitució (veure comentaris al codi):
- **Una sola passada** amb una regex d'alternança ordenada de més llarga a més curta, per
  evitar que una substitució ja aplicada torni a ser capturada.
- **L'entitat objectiu queda exclosa**: la Fase 2 ja l'ha transcrita, i flexionada, cosa
  que un find-and-replace no sap fer.
- Es conserva `tts_text_base` (el valor previ) i s'afegeix `secondary_applied`, per poder
  auditar què s'ha canviat a cada frase.


In [ ]:
# 1. Diccionario final = principal (auditado en el Pas B) + secundario (auditado en 3b).
#    El principal tiene prioridad: es el que se ha revisado con más cuidado.
with open(DICCIONARI_PRINCIPAL, 'r', encoding='utf-8') as f:
    diccionari_principal = json.load(f)

with open(DICCIONARI_SECUNDARI, 'r', encoding='utf-8') as f:
    diccionari_secundari = json.load(f)

diccionari_complet = {**diccionari_secundari, **diccionari_principal}

# Descartamos las identidades (entidades que ya se leen bien tal cual): sustituirlas no
# aporta nada y solo añade riesgo de tocar texto que ya estaba correcto.
diccionari_final = {k: v for k, v in diccionari_complet.items() if clau(k) != clau(v)}

print(f"Diccionario: {len(diccionari_complet)} entradas "
      f"({len(diccionari_principal)} principales + {len(diccionari_secundari)} secundarias), "
      f"{len(diccionari_final)} aplicables tras descartar identidades")


def construir_regex(claus):
    """Una única regex con todas las entidades en alternancia, ordenadas de más larga a
    más corta. Hacerlo en una sola pasada evita el efecto cascada (que una sustitución ya
    aplicada vuelva a ser capturada por otra entidad más corta), y el orden por longitud
    garantiza que gane la coincidencia más específica ('FC Barcelona' antes que 'FC')."""
    patrons = [patro_entitat(c) for c in sorted(claus, key=len, reverse=True)]
    return re.compile("|".join(patrons), re.IGNORECASE)


REGEX_ENTITATS = construir_regex(diccionari_final)
MAPA_FONETIC = {clau(k): v for k, v in diccionari_final.items()}


def aplicar_fonetica(text, exclusions=()):
    """Sustituye en `text` las entidades del diccionario por su forma fonética.
    Devuelve (texto_resultante, entidades_aplicadas)."""
    aplicades = []

    def _substituir(m):
        original = m.group(0)
        k = clau(original)
        if k in exclusions:
            return original
        fonetica = MAPA_FONETIC[k]

        # Preservamos la minúscula inicial cuando la entidad aparece a mitad de frase
        # ('una startup' -> 'una startap'), pero nunca en siglas deletreadas ('U E'),
        # donde las mayúsculas forman parte de la transcripción.
        if original.islower() and not fonetica.isupper():
            fonetica = fonetica[:1].lower() + fonetica[1:]

        # Los símbolos van pegados al número ('30%', '25ºC') pero su transcripción es
        # una palabra, así que hay que separarla o quedaría '30por ciento'.
        inici, fi = m.span()
        if inici > 0 and text[inici - 1].isalnum() and fonetica[:1].isalnum():
            fonetica = " " + fonetica
        if fi < len(text) and text[fi].isalnum() and fonetica[-1:].isalnum():
            fonetica = fonetica + " "

        aplicades.append(original)
        return fonetica

    return REGEX_ENTITATS.sub(_substituir, text), aplicades


# 2. Aplicar a todo el dataset
registres_finals = []
canviats = 0
comptador_aplicades = Counter()

with open(DATASET_IN, 'r', encoding='utf-8') as f:
    for linia in f:
        linia = linia.strip()
        if not linia:
            continue
        registre = json.loads(linia)

        # La entidad objetivo ya la ha transcrito la Fase 2 (y flexionada, cosa que un
        # find-and-replace no sabe hacer): la excluimos para no tocarla dos veces.
        entitat = registre["entity"]
        exclusions = {clau(entitat), clau(diccionari_complet.get(entitat, entitat))}

        text_base = registre["tts_text"]
        text_final, aplicades = aplicar_fonetica(text_base, exclusions=exclusions)

        registre["tts_text_base"] = text_base
        registre["tts_text"] = text_final
        registre["secondary_applied"] = aplicades
        registres_finals.append(registre)

        if aplicades:
            canviats += 1
            comptador_aplicades.update(clau(a) for a in aplicades)

DATASET_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(DATASET_OUT, 'w', encoding='utf-8') as f_out:
    for registre in registres_finals:
        f_out.write(json.dumps(registre, ensure_ascii=False) + "\n")

print(f"\nDataset final: {DATASET_OUT}")
print(f"Frases procesadas: {len(registres_finals)} | con alguna entidad secundaria sustituida: {canviats}")
print("\nEntidades secundarias más aplicadas:")
for ent, n in comptador_aplicades.most_common(15):
    print(f"  {n:5d}x  {ent}")

# 3. Muestra antes/después para verificar a ojo antes de mandarlo al TTS
print("\n--- MUESTRA ---")
for registre in [r for r in registres_finals if r["secondary_applied"]][:5]:
    print(f"\n[{registre['entity']}]")
    print(f"  antes:  {registre['tts_text_base']}")
    print(f"  ahora:  {registre['tts_text']}")
